# Weighted Ensemble Model with Optuna Optimization

This notebook combines the three optimized models (CatBoost, LightGBM, and XGBoost) into a weighted ensemble and uses Optuna to find optimal weights.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
import optuna
from optuna.samplers import TPESampler

# Add parent directory to path to import from helpers
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from helpers.process_data import process_data, preprocess_data, in_fold_process_data

print("Libraries imported successfully!")

Libraries imported successfully!


/Users/winston/Documents/School/Y3S1/BT4012/Kaggle/Competition 1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load training and test data
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

y_train = train['target']

print("Data loaded:")
print(f"Training data shape: {train.shape}")
print(f"Test data shape: {test.shape}")
print("\nTarget distribution:")
print(y_train.value_counts().sort_index())

Data loaded:
Training data shape: (26206, 19)
Test data shape: (11232, 19)

Target distribution:
target
0    17423
1     8783
Name: count, dtype: int64


## Load Pre-trained Optimized Models

In [3]:
# Load the three optimized models
catboost_model = joblib.load('../models/trained/catboost_model_optuna.joblib')
lgb_model = joblib.load('../models/trained/lgb_model_optuna.joblib')
xgb_model = joblib.load('../models/trained/xgb_model_optuna.joblib')

print("✓ CatBoost Optuna model loaded")
print("✓ LightGBM Optuna model loaded")
print("✓ XGBoost Optuna model loaded")

✓ CatBoost Optuna model loaded
✓ LightGBM Optuna model loaded
✓ XGBoost Optuna model loaded


## Prepare Data for Ensemble

In [4]:
# Preprocess data (same as in individual models)
raw_train_features = train.drop('target', axis=1)
X_preprocessed = preprocess_data(raw_train_features, verbose=True, include_statuses_count=True)

print(f"\nPreprocessed training data shape: {X_preprocessed.shape}")

=== Pre-CV Data Preprocessing (No Leakage) ===
Original dataframe shape: (26206, 18)

Extracting numerical features...
  - favourites_count: extracted (missing: 0)
  - followers_count: extracted (missing: 0)
  - friends_count: extracted (missing: 0)
  - average_tweets_per_day: extracted (missing: 0)
  - account_age_days: extracted (missing: 0)
  - statuses_count: extracted (missing: 0)

Extracting binary features...
  - default_profile: extracted (missing: 0)
  - default_profile_image: extracted (missing: 0)
  - geo_enabled: extracted (missing: 0)
  - verified: extracted (missing: 0)

Extracting categorical features...
  - lang: extracted (49 categories)

Extracting textual features...
  - location_len: mean=11.05
  - description_len: mean=66.79
  - screen_name_len: mean=11.12
  - profile_background_image_url_len: mean=42.25
  - profile_image_url_len: mean=75.52
  - description_has_url: 3613 accounts (13.8%)
  - screen_name_has_numbers: 7015 accounts (26.8%)
  - location_is_empty: 9219

## Weighted Ensemble with Optuna Optimization

In [5]:
# Define weighted ensemble function
def weighted_ensemble_predict_proba(catboost_proba, lgb_proba, xgb_proba, w1, w2, w3):
    """
    Combine predictions from three models with weights
    
    Args:
        catboost_proba: CatBoost predictions
        lgb_proba: LightGBM predictions  
        xgb_proba: XGBoost predictions
        w1, w2, w3: Weights for each model (will be normalized)
    
    Returns:
        Weighted ensemble predictions
    """
    # Normalize weights to sum to 1
    total = w1 + w2 + w3
    w1_norm = w1 / total
    w2_norm = w2 / total
    w3_norm = w3 / total
    
    # Combine predictions
    ensemble_proba = w1_norm * catboost_proba + w2_norm * lgb_proba + w3_norm * xgb_proba
    return ensemble_proba

print("Weighted ensemble function defined")

Weighted ensemble function defined


In [12]:
# Prepare full training data with same preprocessing as used in individual models
print("Preparing full training data for predictions...")
X_train_full_processed, _, impute_vals_train = in_fold_process_data(
    X_preprocessed,
    X_preprocessed,  # Use same data for both to get consistent shape
    verbose=True
)

# Get predictions from all three models on training data
print("\nGenerating model predictions on training data...")
catboost_train_proba = catboost_model.predict_proba(X_train_full_processed)[:, 1]
lgb_train_proba = lgb_model.predict_proba(X_train_full_processed)[:, 1]
xgb_train_proba = xgb_model.predict_proba(X_train_full_processed)[:, 1]
print("✓ All model predictions generated")

# Optuna objective function - optimize on training predictions with CV
def objective(trial):
    """
    Optuna objective function to optimize ensemble weights using CV on training predictions
    """
    # Suggest weights for each model
    w_catboost = trial.suggest_float('w_catboost', 0.0, 1.0)
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    
    # Use stratified k-fold cross-validation on the model predictions
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_preprocessed, y_train)):
        # Get predictions for this fold
        catboost_fold_proba = catboost_train_proba[val_idx]
        lgb_fold_proba = lgb_train_proba[val_idx]
        xgb_fold_proba = xgb_train_proba[val_idx]
        y_val_fold = y_train.iloc[val_idx]
        
        # Ensemble predictions
        ensemble_val_proba = weighted_ensemble_predict_proba(
            catboost_fold_proba, lgb_fold_proba, xgb_fold_proba,
            w_catboost, w_lgb, w_xgb
        )
        
        # Calculate accuracy
        ensemble_val_pred = (ensemble_val_proba >= 0.5).astype(int)
        fold_score = accuracy_score(y_val_fold, ensemble_val_pred)
        cv_scores.append(fold_score)
    
    # Return mean CV score
    mean_cv_score = np.mean(cv_scores)
    return mean_cv_score

print("\nOptuna objective function defined")

Preparing full training data for predictions...
=== In-Fold Processing (Fit on Train, Transform Train & Val) ===
Train shape: (26206, 33), Val shape: (26206, 33)
Numerical scaling: DISABLED

Imputing numerical features...

Imputing binary features...

Encoding categorical features...
  - lang: grouped 0 categories with freq < 0
  - lang: one-hot encoded into 49 columns

Final data validation...
  ✓ All columns are numeric

=== In-Fold Processing Complete ===
Train shape: (26206, 81), Val shape: (26206, 81)

Generating model predictions on training data...
✓ All model predictions generated

Optuna objective function defined


In [13]:
# Run Optuna optimization
print("="*70)
print("RUNNING OPTUNA OPTIMIZATION FOR ENSEMBLE WEIGHTS")
print("="*70)
print("\nOptimizing ensemble weights using 5-fold cross-validation...")
print("This may take a while...\n")

# Create Optuna study
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42)
)

# Optimize
study.optimize(objective, n_trials=100, show_progress_bar=True)

# Get best parameters
best_params = study.best_params
best_score = study.best_value

print("\n" + "="*70)
print("OPTIMIZATION COMPLETE")
print("="*70)
print(f"\nBest CV Accuracy: {best_score:.6f}")
print("\nBest Weights:")
print(f"  CatBoost: {best_params['w_catboost']:.4f}")
print(f"  LightGBM: {best_params['w_lgb']:.4f}")
print(f"  XGBoost:  {best_params['w_xgb']:.4f}")

# Normalize weights
total_weight = best_params['w_catboost'] + best_params['w_lgb'] + best_params['w_xgb']
print("\nNormalized Weights (sum to 1.0):")
print(f"  CatBoost: {best_params['w_catboost']/total_weight:.4f}")
print(f"  LightGBM: {best_params['w_lgb']/total_weight:.4f}")
print(f"  XGBoost:  {best_params['w_xgb']/total_weight:.4f}")
print("="*70)

[I 2025-10-07 16:12:27,572] A new study created in memory with name: no-name-0dca501a-cb58-4c9a-ad80-5cdee5222dbc


RUNNING OPTUNA OPTIMIZATION FOR ENSEMBLE WEIGHTS

Optimizing ensemble weights using 5-fold cross-validation...
This may take a while...



Best trial: 23. Best value: 0.931504:  25%|██▌       | 25/100 [00:00<00:00, 125.24it/s]

[I 2025-10-07 16:12:27,583] Trial 0 finished with value: 0.9294437855021682 and parameters: {'w_catboost': 0.3745401188473625, 'w_lgb': 0.9507143064099162, 'w_xgb': 0.7319939418114051}. Best is trial 0 with value: 0.9294437855021682.
[I 2025-10-07 16:12:27,593] Trial 1 finished with value: 0.9263911077080522 and parameters: {'w_catboost': 0.5986584841970366, 'w_lgb': 0.15601864044243652, 'w_xgb': 0.15599452033620265}. Best is trial 0 with value: 0.9294437855021682.
[I 2025-10-07 16:12:27,601] Trial 2 finished with value: 0.9302832762634239 and parameters: {'w_catboost': 0.05808361216819946, 'w_lgb': 0.8661761457749352, 'w_xgb': 0.6011150117432088}. Best is trial 2 with value: 0.9302832762634239.
[I 2025-10-07 16:12:27,608] Trial 3 finished with value: 0.9303214805985238 and parameters: {'w_catboost': 0.7080725777960455, 'w_lgb': 0.020584494295802447, 'w_xgb': 0.9699098521619943}. Best is trial 3 with value: 0.9303214805985238.
[I 2025-10-07 16:12:27,614] Trial 4 finished with value: 0.

Best trial: 23. Best value: 0.931504:  27%|██▋       | 27/100 [00:00<00:00, 125.24it/s]

...

[I 2025-10-07 16:12:28,020] Trial 55 finished with value: 0.933641282987183 and parameters: {'w_catboost': 0.037990903009925955, 'w_lgb': 0.05626773808717271, 'w_xgb': 0.5675966953048251}. Best is trial 55 with value: 0.933641282987183.
[I 2025-10-07 16:12:28,028] Trial 56 finished with value: 0.933526801018093 and parameters: {'w_catboost': 0.03593010541680753, 'w_lgb': 0.052878487414207365, 'w_xgb': 0.5779264017129179}. Best is trial 55 with value: 0.933641282987183.
[I 2025-10-07 16:12:28,035] Trial 57 finished with value: 0.9326872884174693 and parameters: {'w_catboost': 0.06485114180221298, 'w_lgb': 0.0776419050087187, 'w_xgb': 0.45625511809141484}. Best is trial 55 with value: 0.933641282987183.
[I 2025-10-07 16:12:28,044] Trial 58 finished with value: 0.9329925736683753 and parameters: {'w_catboost': 0.04496050673069847, 'w_lgb': 0.047302144373362595, 'w_xgb': 0.3270870167652243}. Best is trial 55 with value: 0.933641282987183.
[I 2025-10-07 16:12:28,052] Trial 59 finished with 

Best trial: 75. Best value: 0.934748:  80%|████████  | 80/100 [00:00<00:00, 126.70it/s]

[I 2025-10-07 16:12:28,181] Trial 76 finished with value: 0.932992580948165 and parameters: {'w_catboost': 0.0042662340311644745, 'w_lgb': 0.11324789900799839, 'w_xgb': 0.425224641951929}. Best is trial 75 with value: 0.9347479129025607.
[I 2025-10-07 16:12:28,189] Trial 77 finished with value: 0.9325346894707527 and parameters: {'w_catboost': 0.09566520261778916, 'w_lgb': 0.08253371749734086, 'w_xgb': 0.424461847095818}. Best is trial 75 with value: 0.9347479129025607.
[I 2025-10-07 16:12:28,198] Trial 78 finished with value: 0.9339084148615153 and parameters: {'w_catboost': 0.0018891359260611038, 'w_lgb': 0.026542723596779377, 'w_xgb': 0.36135321448027624}. Best is trial 75 with value: 0.9347479129025607.
[I 2025-10-07 16:12:28,206] Trial 79 finished with value: 0.9328017776663484 and parameters: {'w_catboost': 0.11686753532108321, 'w_lgb': 0.0008835923921878407, 'w_xgb': 0.4521791369205843}. Best is trial 75 with value: 0.9347479129025607.
[I 2025-10-07 16:12:28,214] Trial 80 finish

Best trial: 75. Best value: 0.934748: 100%|██████████| 100/100 [00:00<00:00, 126.32it/s]

[I 2025-10-07 16:12:28,222] Trial 81 finished with value: 0.9339465755178787 and parameters: {'w_catboost': 0.0013558642044348165, 'w_lgb': 0.028431987459287468, 'w_xgb': 0.4099565775406949}. Best is trial 75 with value: 0.9347479129025607.
[I 2025-10-07 16:12:28,230] Trial 82 finished with value: 0.9331070483576758 and parameters: {'w_catboost': 0.07557233355474378, 'w_lgb': 0.021891514820133406, 'w_xgb': 0.41064629681611875}. Best is trial 75 with value: 0.9347479129025607.
[I 2025-10-07 16:12:28,238] Trial 83 finished with value: 0.9332596982629184 and parameters: {'w_catboost': 0.018417927963431082, 'w_lgb': 0.08153320241438827, 'w_xgb': 0.5048572002885269}. Best is trial 75 with value: 0.9347479129025607.
[I 2025-10-07 16:12:28,246] Trial 84 finished with value: 0.9314662202117386 and parameters: {'w_catboost': 0.12105963762349559, 'w_lgb': 0.1129239291058065, 'w_xgb': 0.37658619517303504}. Best is trial 75 with value: 0.9347479129025607.
[I 2025-10-07 16:12:28,255] Trial 85 finis

## Generate Predictions on Test Set

In [14]:
# Prepare test data
print("="*70)
print("PREPARING TEST DATA")
print("="*70)

# Preprocess test data
X_test_preprocessed = preprocess_data(test, verbose=True, include_statuses_count=True)

# Process full training data to get imputation values and process test data
X_train_full_processed, X_test_processed, impute_vals_full = in_fold_process_data(
    X_preprocessed,
    X_test_preprocessed,
    verbose=True
)

print(f"\nTest data shape after processing: {X_test_processed.shape}")
print("="*70)

PREPARING TEST DATA
=== Pre-CV Data Preprocessing (No Leakage) ===
Original dataframe shape: (11232, 19)

Extracting numerical features...
  - favourites_count: extracted (missing: 0)
  - followers_count: extracted (missing: 0)
  - friends_count: extracted (missing: 0)
  - average_tweets_per_day: extracted (missing: 0)
  - account_age_days: extracted (missing: 0)
  - statuses_count: extracted (missing: 0)

Extracting binary features...
  - default_profile: extracted (missing: 0)
  - default_profile_image: extracted (missing: 0)
  - geo_enabled: extracted (missing: 0)
  - verified: extracted (missing: 0)

Extracting categorical features...
  - lang: extracted (47 categories)

Extracting textual features...
  - location_len: mean=11.20
  - description_len: mean=67.09
  - screen_name_len: mean=11.08
  - profile_background_image_url_len: mean=42.77
  - profile_image_url_len: mean=75.43
  - description_has_url: 1532 accounts (13.6%)
  - screen_name_has_numbers: 2966 accounts (26.4%)
  - loc

In [15]:
# Get predictions from each model on test set
print("="*70)
print("GENERATING ENSEMBLE PREDICTIONS")
print("="*70)

catboost_test_proba = catboost_model.predict_proba(X_test_processed)[:, 1]
lgb_test_proba = lgb_model.predict_proba(X_test_processed)[:, 1]
xgb_test_proba = xgb_model.predict_proba(X_test_processed)[:, 1]

print("✓ CatBoost predictions generated")
print("✓ LightGBM predictions generated")
print("✓ XGBoost predictions generated")

# Create weighted ensemble predictions using optimal weights
ensemble_test_proba = weighted_ensemble_predict_proba(
    catboost_test_proba,
    lgb_test_proba,
    xgb_test_proba,
    best_params['w_catboost'],
    best_params['w_lgb'],
    best_params['w_xgb']
)

# Convert probabilities to class predictions
ensemble_test_pred = (ensemble_test_proba >= 0.5).astype(int)

print("\n✓ Ensemble predictions generated")
print(f"\nPrediction distribution:")
print(pd.Series(ensemble_test_pred).value_counts().sort_index())
print("="*70)

GENERATING ENSEMBLE PREDICTIONS
✓ CatBoost predictions generated
✓ LightGBM predictions generated
✓ XGBoost predictions generated

✓ Ensemble predictions generated

Prediction distribution:
0    7953
1    3279
Name: count, dtype: int64


## Create Submission File

In [22]:
# Create submission DataFrame with PROBABILITIES (for ROC AUC scoring)
# Note: index should be sequential integers (0, 1, 2, ...), not test IDs
submission = pd.DataFrame({
    'index': range(len(ensemble_test_proba)),
    'target': ensemble_test_proba  # Use probabilities instead of binary predictions
})

# Save to CSV
submission_path = '../submissions/weighted_ensemble_optuna.csv'
submission.to_csv(submission_path, index=False)

print("="*70)
print("SUBMISSION FILE CREATED")
print("="*70)
print(f"\nSubmission saved to: {submission_path}")
print(f"Submission shape: {submission.shape}")
print("\nFirst few rows:")
print(submission.head())
print("\nSubmission statistics:")
print(f"  Min probability: {submission['target'].min():.6f}")
print(f"  Max probability: {submission['target'].max():.6f}")
print(f"  Mean probability: {submission['target'].mean():.6f}")
print(f"  Median probability: {submission['target'].median():.6f}")
print("="*70)

SUBMISSION FILE CREATED

Submission saved to: ../submissions/weighted_ensemble_optuna.csv
Submission shape: (11232, 2)

First few rows:
   index    target
0      0  0.014481
1      1  0.908142
2      2  0.060851
3      3  0.047791
4      4  0.992869

Submission statistics:
  Min probability: 0.000739
  Max probability: 0.999202
  Mean probability: 0.325740
  Median probability: 0.113258


## Save Ensemble Configuration

In [17]:
# Save the ensemble configuration
ensemble_config = {
    'best_weights': best_params,
    'normalized_weights': {
        'w_catboost': best_params['w_catboost']/total_weight,
        'w_lgb': best_params['w_lgb']/total_weight,
        'w_xgb': best_params['w_xgb']/total_weight
    },
    'best_cv_score': best_score,
    'study': study
}

# Save configuration
config_path = '../models/trained/ensemble_config_optuna.joblib'
joblib.dump(ensemble_config, config_path)

print(f"✓ Ensemble configuration saved to: {config_path}")

✓ Ensemble configuration saved to: ../models/trained/ensemble_config_optuna.joblib


## Summary

In [21]:
print("="*70)
print("WEIGHTED ENSEMBLE SUMMARY")
print("="*70)
print("\n📊 Models Combined:")
print("   1. CatBoost (Optuna-optimized)")
print("   2. LightGBM (Optuna-optimized)")
print("   3. XGBoost (Optuna-optimized)")

print("\n🎯 Best Ensemble Weights (from 100 Optuna trials):")
print(f"   CatBoost: {best_params['w_catboost']:.4f} → {best_params['w_catboost']/total_weight:.4f} (normalized)")
print(f"   LightGBM: {best_params['w_lgb']:.4f} → {best_params['w_lgb']/total_weight:.4f} (normalized)")
print(f"   XGBoost:  {best_params['w_xgb']:.4f} → {best_params['w_xgb']/total_weight:.4f} (normalized)")

print(f"\n📈 Cross-Validation Accuracy: {best_score:.6f}")

print("\n📁 Files Created:")
print(f"   ✓ Submission: submissions/weighted_ensemble_optuna.csv")
print(f"   ✓ Config: models/trained/ensemble_config_optuna.joblib")

print("\n🔢 Submission Statistics:")
print(f"   Total predictions: {len(submission)}")
print(f"   Probabilities range: [{submission['target'].min():.4f}, {submission['target'].max():.4f}]")
print(f"   Mean probability: {submission['target'].mean():.4f}")
print(f"   Predictions > 0.5 (Bot): {(submission['target'] > 0.5).sum()} ({(submission['target'] > 0.5).sum()/len(submission)*100:.1f}%)")
print(f"   Predictions ≤ 0.5 (Human): {(submission['target'] <= 0.5).sum()} ({(submission['target'] <= 0.5).sum()/len(submission)*100:.1f}%)")

print("\n" + "="*70)
print("✅ Weighted ensemble complete! Ready for submission to Kaggle.")
print("   (Submitting probabilities for ROC AUC scoring)")
print("="*70)

WEIGHTED ENSEMBLE SUMMARY

📊 Models Combined:
   1. CatBoost (Optuna-optimized)
   2. LightGBM (Optuna-optimized)
   3. XGBoost (Optuna-optimized)

🎯 Best Ensemble Weights (from 100 Optuna trials):
   CatBoost: 0.0010 → 0.0024 (normalized)
   LightGBM: 0.0001 → 0.0003 (normalized)
   XGBoost:  0.4214 → 0.9972 (normalized)

📈 Cross-Validation Accuracy: 0.934748

📁 Files Created:
   ✓ Submission: submissions/weighted_ensemble_optuna.csv
   ✓ Config: models/trained/ensemble_config_optuna.joblib

🔢 Submission Statistics:
   Total predictions: 11232
   Probabilities range: [0.0007, 0.9992]
   Mean probability: 0.3257
   Predictions > 0.5 (Bot): 3279 (29.2%)
   Predictions ≤ 0.5 (Human): 7953 (70.8%)

✅ Weighted ensemble complete! Ready for submission to Kaggle.
   (Submitting probabilities for ROC AUC scoring)
